In [4]:
import pandas as pd

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [3]:
billing_path = '/content/billings_preprocessed.csv'
calls_path = '/content/cc_calls.csv'

In [6]:
billings = pd.read_csv(billing_path)
cc_calls = pd.read_csv(calls_path)

/tmp/ipykernel_3398/3079290077.py:1: DtypeWarning: Columns (50,51) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv(billing_path)


In [7]:
billings.shape

(113291, 61)

In [8]:
cc_calls.shape

(32882, 33)

In [9]:
# Convert Prospect_Renewal_Date (already datetime in your summary)
billings['Prospect_Renewal_Date'] = pd.to_datetime(billings['Prospect_Renewal_Date'], errors='coerce')

# Convert Call_Date from string "DD-MM-YYYY" to datetime
cc_calls['Call_Date'] = pd.to_datetime(cc_calls['Call_Date'], format='%d-%m-%Y', errors='coerce')

In [10]:
merged = pd.merge(
    cc_calls,
    billings[['Co_Ref', 'Prospect_Renewal_Date']],  # Only bring the needed column for now
    on='Co_Ref',
    how='left'   # left join so we keep all calls (we can drop later if needed)
)

print(f"After join shape: {merged.shape}")

After join shape: (90625, 34)


In [11]:
# Calculate days to renewal
merged['days_to_renewal'] = (merged['Prospect_Renewal_Date'] - merged['Call_Date']).dt.days

# Strict filter: calls made 0 to 14 days before renewal (inclusive)
# Change to (merged['days_to_renewal'] > 0) & ... if you want strictly before renewal date
filtered_cc = merged[
    (merged['days_to_renewal'] >= 0) &
    (merged['days_to_renewal'] <= 14) &
    (merged['Call_Date'].notna()) &
    (merged['Prospect_Renewal_Date'].notna())
].copy()

print(f"Final filtered shape (14-days pre-renewal): {filtered_cc.shape}")
print(f"Unique contractors in this window: {filtered_cc['Co_Ref'].nunique()}")

Final filtered shape (14-days pre-renewal): (2123, 35)
Unique contractors in this window: 1834


In [15]:
keep_columns = [
    'Co_Ref',
    'Prospect_Renewal_Date',      # Keep this - very useful
    'days_to_renewal',            # Keep this - important feature
    'Call_Date',
    'Direction',
    'cc_care_package',
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_call_initiated_by',
    'cc_questionnaire_completion',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_contractor_sentiment',
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained',
    'Analysed_Call',
    'Call_Year'
]

# Drop everything else (especially all extra billing columns)
df_cc_calls = filtered_cc[keep_columns].copy()

print(f"Final dataset after column dropping: {df_cc_calls.shape}")
print("\nColumns kept:")
print(df_cc_calls.columns.tolist())


Final dataset after column dropping: (2123, 34)

Columns kept:
['Co_Ref', 'Prospect_Renewal_Date', 'days_to_renewal', 'Call_Date', 'Direction', 'cc_care_package', 'cc_care_package_discussed', 'cc_urgency_getting_on_site', 'cc_external_consultant', 'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship', 'cc_call_initiated_by', 'cc_questionnaire_completion', 'cc_chasing_response', 'cc_issues_within_questionnaire', 'cc_login_issues', 'cc_platform_issues', 'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns', 'cc_questions_harder_than_expected', 'cc_dissatisfaction_support', 'cc_contractor_sentiment', 'cc_contractor_sentiment_start_score', 'cc_contractor_sentiment_end_score', 'cc_contractor_sentiment_overall_score', 'cc_contractor_sentiment_issues_score', 'cc_pricing_mentioned', 'cc_pricing_sentiment_impact', 'cc_refund_discussed', 'cc_contractor_suggest_leave', 'cc_contractor_complained', 'Analysed_Call', 'Call_Year']


In [17]:
df_cc_calls.to_csv('df_cc_calls.csv', index=False)

from google.colab import files
files.download('df_cc_calls.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
df_cc_calls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2123 entries, 3 to 90596
Data columns (total 34 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    2123 non-null   object        
 1   Prospect_Renewal_Date                     2123 non-null   datetime64[ns]
 2   days_to_renewal                           2123 non-null   float64       
 3   Call_Date                                 2123 non-null   datetime64[ns]
 4   Direction                                 2123 non-null   object        
 5   cc_care_package                           2110 non-null   object        
 6   cc_care_package_discussed                 2110 non-null   object        
 7   cc_urgency_getting_on_site                2110 non-null   object        
 8   cc_external_consultant                    2110 non-null   object        
 9   cc_agent_cross_sell_attempt       

In [14]:
df_cc_calls.describe(include='all')

,Co_Ref,Prospect_Renewal_Date,days_to_renewal,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_call_initiated_by,cc_questionnaire_completion,cc_chasing_response,cc_issues_within_questionnaire,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_contractor_sentiment,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Analysed_Call,Call_Year
count,2123,2123,2123.000000,2123,2123,2110,2110,2110,2110,2110,2110,2110,2110,2122,2122,2063,2122,2122,2122,2122,2122,2122,2120,2120,2120,2120,2120,2120,2120,2120,2120,2120,2123.0,2123.000000
unique,1834,NaN,NaN,NaN,2,11,3,3,3,5,4,3,5,3,12,12,3,3,3,3,3,3,6,11,15,19,17,4,4,3,3,3,NaN,NaN
top,AA0994,NaN,NaN,NaN,OUT_BOUND,Not Discussed,No,No,No,No,No,No,Customer,No,No,No,No,No,No,No,No,No,Satisfied,50,90,90,Not Discussed,No,No,No,No,No,NaN,NaN
freq,7,NaN,NaN,NaN,1510,1652,1649,1931,1864,2047,1863,2012,1487,1734,1645,1754,2028,1865,2050,1913,2114,2069,1080,1678,929,715,1305,1761,1990,2086,2010,1936,NaN,NaN
mean,NaN,2025-06-23 18:07:17.494112,6.830900,2025-06-16 22:10:47.762599936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2024.913801
min,NaN,2024-09-13 00:00:00,0.000000,2024-09-13 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2024.000000
25%,NaN,2025-03-08 00:00:00,3.000000,2025-03-03 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2025.000000
50%,NaN,2025-07-09 00:00:00,7.000000,2025-07-02 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2025.000000
75%,NaN,2025-10-09 00:00:00,10.500000,2025-10-02 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2025.000000
max,NaN,2026-01-21 00:00:00,14.000000,2026-01-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2026.000000


# Task
Inspect and display the unique values and their counts for all columns in `df_cc_calls` that have the 'object' data type.

## Inspect Object Columns

### Subtask:
Examine the unique values and their counts for all columns with 'object' data type in `df_cc_calls`. This will help identify inconsistencies, different representations of similar values, and the presence of missing data, which are crucial before type conversion or further cleaning.


**Reasoning**:
To inspect the 'object' type columns, I need to first identify them, then iterate through each of these columns to display their unique values and counts, including any missing values, as requested in the instructions.



In [18]:
object_cols = df_cc_calls.select_dtypes(include='object').columns

for col in object_cols:
    print(f"\n--- Column: {col} ---")
    print(df_cc_calls[col].value_counts(dropna=False))



--- Column: Co_Ref ---
Co_Ref
AA0994    7
SN4133    6
EQ2869    5
WL4087    4
NC5321    4
         ..
NI6323    1
NM4287    1
KF1439    1
GB7506    1
DT7731    1
Name: count, Length: 1834, dtype: int64

--- Column: Direction ---
Direction
OUT_BOUND    1510
IN_BOUND      613
Name: count, dtype: int64

--- Column: cc_care_package ---
cc_care_package
Not Discussed                                          1652
Express                                                 199
Standard                                                195
Premier                                                  46
NaN                                                      13
Assisted                                                 11
[Standard/Premier/Express/Not Discussed]                  2
Assisted (currently), Standard (requested)                1
Funder Package (initially), then Express (upgraded)       1
Clickers package                                          1
Class                                            

In [19]:
yes_no_cols = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_call_initiated_by',
    'cc_questionnaire_completion',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained'
]

print(f"Defined {len(yes_no_cols)} Yes/No columns.")

Defined 21 Yes/No columns.




```python
# Define mapping for cc_care_package
# Inconsistent or unclear values are mapped to 'Not Discussed',
# while clear upgrades are mapped to their intended package.
care_package_mapping = {
    'Standard': 'Standard',
    'Express': 'Express',
    'Premier': 'Premier',
    'Not Discussed': 'Not Discussed',
    'Assisted': 'Not Discussed',
    '[Standard/Premier/Express/Not Discussed]': 'Not Discussed',
    'Assisted (currently), Standard (requested)': 'Not Discussed',
    'Clickers package': 'Not Discussed',
    'Class': 'Not Discussed',
    'Primary': 'Not Discussed',
    'Funder Package (initially), then Express (upgraded)': 'Express'
}

# Apply the mapping
df_cc_calls['cc_care_package'] = df_cc_calls['cc_care_package'].replace(care_package_mapping)

# Fill any remaining NaN values with 'Not Discussed'
df_cc_calls['cc_care_package'].fillna('Not Discussed', inplace=True)

# Verify the cleaning by displaying unique values and their counts
print("\n--- 'cc_care_package' column after mapping and filling NaNs ---")
print(df_cc_calls['cc_care_package'].value_counts(dropna=False))
```

In [20]:
care_package_mapping = {
    'Standard': 'Standard',
    'Express': 'Express',
    'Premier': 'Premier',
    'Not Discussed': 'Not Discussed',
    'Assisted': 'Not Discussed',
    '[Standard/Premier/Express/Not Discussed]': 'Not Discussed',
    'Assisted (currently), Standard (requested)': 'Not Discussed',
    'Clickers package': 'Not Discussed',
    'Class': 'Not Discussed',
    'Primary': 'Not Discussed',
    'Funder Package (initially), then Express (upgraded)': 'Express'
}
df_cc_calls['cc_care_package'] = df_cc_calls['cc_care_package'].replace(care_package_mapping)

In [21]:
df_cc_calls['cc_care_package'].fillna('Not Discussed', inplace=True)

/tmp/ipykernel_3398/3983021724.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cc_calls['cc_care_package'].fillna('Not Discussed', inplace=True)


In [24]:
print("\n--- 'cc_care_package' column after mapping and filling NaNs ---")
print(df_cc_calls['cc_care_package'].value_counts(dropna=False))


--- 'cc_care_package' column after mapping and filling NaNs ---
cc_care_package
Not Discussed    1682
Express           200
Standard          195
Premier            46
Name: count, dtype: int64


In [27]:
yes_no_columns_to_standardize = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_questions_harder_than_expected',
    'cc_process_complexity_concerns',
    'cc_dissatisfaction_support'
]

print("Columns to standardize to 'Yes', 'No', or 'Not Mentioned':")
print(yes_no_columns_to_standardize)

Columns to standardize to 'Yes', 'No', or 'Not Mentioned':
['cc_care_package_discussed', 'cc_urgency_getting_on_site', 'cc_external_consultant', 'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship', 'cc_questionnaire_completion', 'cc_login_issues', 'cc_platform_issues', 'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns', 'cc_questions_harder_than_expected', 'cc_dissatisfaction_support', 'cc_pricing_mentioned', 'cc_pricing_sentiment_impact', 'cc_refund_discussed', 'cc_contractor_suggest_leave', 'cc_contractor_complained']


In [26]:
for col in yes_no_columns_to_standardize:
    df_cc_calls[col] = df_cc_calls[col].astype(str).str.upper()
    df_cc_calls[col] = df_cc_calls[col].replace({
        'YES': 'Yes',
        'NO': 'No',
        'NAN': 'Not Mentioned',
        '[YES/NO]': 'Not Mentioned'
    })

    df_cc_calls.loc[~df_cc_calls[col].isin(['Yes', 'No']), col] = 'Not Mentioned'
print("\n--- 'cc_care_package_discussed' column after standardization ---")
print(df_cc_calls['cc_care_package_discussed'].value_counts(dropna=False))



--- 'cc_care_package_discussed' column after standardization ---
cc_care_package_discussed
No               1649
Yes               459
Not Mentioned      15
Name: count, dtype: int64

--- 'cc_urgency_getting_on_site' column after standardization ---
cc_urgency_getting_on_site
No               1931
Yes               177
Not Mentioned      15
Name: count, dtype: int64

--- 'cc_dissatisfaction_support' column after standardization ---
cc_dissatisfaction_support
No               2069
Yes                52
Not Mentioned       2
Name: count, dtype: int64


In [28]:
col = 'cc_agent_cross_sell_attempt'

df_cc_calls[col] = df_cc_calls[col].astype(str).str.upper()
df_cc_calls[col] = df_cc_calls[col].replace({
    'YES': 'Yes',
    'NO': 'No'
})

df_cc_calls[col] = df_cc_calls[col].replace({
    'NAN': 'Not Mentioned',
    '[YES/NO]': 'Not Mentioned'
})

df_cc_calls.loc[~df_cc_calls[col].isin(['Yes', 'No', 'Not Mentioned']), col] = 'Yes'
print(f"\n--- '{col}' column after standardization ---")
print(df_cc_calls[col].value_counts(dropna=False))


--- 'cc_agent_cross_sell_attempt' column after standardization ---
cc_agent_cross_sell_attempt
No               2047
Yes                61
Not Mentioned      15
Name: count, dtype: int64


In [29]:
col = 'cc_call_initiated_by'

df_cc_calls[col] = df_cc_calls[col].astype(str).str.upper()

df_cc_calls[col] = df_cc_calls[col].replace({
    'CUSTOMER': 'Customer',
    'AGENT': 'Agent'
})

df_cc_calls[col] = df_cc_calls[col].replace({'NAN': 'Not Relevant'})

df_cc_calls.loc[~df_cc_calls[col].isin(['Customer', 'Agent']), col] = 'Not Relevant'

print(f"\n--- '{col}' column after standardization ---")
print(df_cc_calls[col].value_counts(dropna=False))


--- 'cc_call_initiated_by' column after standardization ---
cc_call_initiated_by
Customer        1487
Agent            534
Not Relevant     102
Name: count, dtype: int64


In [30]:
col = 'cc_chasing_response'

df_cc_calls[col] = df_cc_calls[col].astype(str).str.upper()
df_cc_calls[col] = df_cc_calls[col].replace({
    'NAN': 'No',
    '[YES/NO]': 'No'
})
df_cc_calls[col] = df_cc_calls[col].replace({
    'YES': 'Yes',
    'NO': 'No'
})

df_cc_calls.loc[~df_cc_calls[col].isin(['Yes', 'No']), col] = 'Yes'

print(f"\n--- '{col}' column after standardization ---")
print(df_cc_calls[col].value_counts(dropna=False))


--- 'cc_chasing_response' column after standardization ---
cc_chasing_response
No     1647
Yes     476
Name: count, dtype: int64


In [31]:
col = 'cc_issues_within_questionnaire'

df_cc_calls[col] = df_cc_calls[col].astype(str).str.upper()
df_cc_calls[col] = df_cc_calls[col].replace({
    'NAN': 'Not Mentioned',
    '[YES/NO]': 'Not Mentioned'
})

df_cc_calls[col] = df_cc_calls[col].replace({
    'YES': 'Yes'
})

# 4. For any values that are not 'Yes' or 'Not Mentioned' after the previous replacements, map them to 'No'
df_cc_calls.loc[~df_cc_calls[col].isin(['Yes', 'Not Mentioned']), col] = 'No'

# 5. Print the unique values and their counts
print(f"\n--- '{col}' column after standardization ---")
print(df_cc_calls[col].value_counts(dropna=False))


--- 'cc_issues_within_questionnaire' column after standardization ---
cc_issues_within_questionnaire
No     1865
Yes     258
Name: count, dtype: int64


In [45]:
import pandas as pd
import numpy as np

# Use your actual dataframe name
df = df_cc_calls.copy()

print("Original shape:", df.shape)


def get_sentiment_from_scores(row):
    """Impute sentiment based on average of the three main scores"""
    scores = []
    for col in ['cc_contractor_sentiment_start_score',
                'cc_contractor_sentiment_end_score',
                'cc_contractor_sentiment_overall_score']:
        val = row[col]
        if pd.notna(val):
            try:
                num = float(val)
                if 0 <= num <= 100:
                    scores.append(num)
            except:
                pass
    if len(scores) == 0:
        return "Neutral"  # safe default

    avg_score = np.mean(scores)

    if avg_score >= 70:
        return "Satisfied"
    elif avg_score >= 40:
        return "Neutral"
    else:
        return "Dissatisfied"


def get_score_from_sentiment(sentiment):
    """Map clean sentiment to reasonable default score"""
    mapping = {
        "Satisfied": 80,
        "Neutral": 50,
        "Dissatisfied": 20
    }
    return mapping.get(sentiment, 50)


# First, clean the column
df['cc_contractor_sentiment_clean'] = df['cc_contractor_sentiment'].astype(str).str.strip()

# Rows that are NOT in the 3 desired categories
valid_sentiments = ['Satisfied', 'Neutral', 'Dissatisfied']
mask_needs_impute = ~df['cc_contractor_sentiment_clean'].isin(valid_sentiments)

print(f"Rows where sentiment needs imputation (based on scores): {mask_needs_impute.sum()}")

# Impute sentiment using scores for messy rows
df.loc[mask_needs_impute, 'cc_contractor_sentiment_clean'] = df.loc[mask_needs_impute].apply(
    get_sentiment_from_scores, axis=1
)

# Fill any remaining NaN with Neutral
df['cc_contractor_sentiment_clean'] = df['cc_contractor_sentiment_clean'].fillna("Neutral")

print("\nFinal Sentiment Distribution (only 3 categories):")
print(df['cc_contractor_sentiment_clean'].value_counts(dropna=False))


# ====================== 3. Clean and Impute All Score Columns ======================
score_columns = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

for score_col in score_columns:
    def clean_score(row):
        val = row[score_col]
        sentiment = row['cc_contractor_sentiment_clean']

        if pd.notna(val):
            try:
                num = float(val)
                if 0 <= num <= 100:
                    return num
            except:
                pass

        return get_score_from_sentiment(sentiment)

    df[f'{score_col}_clean'] = df.apply(clean_score, axis=1)


print("\n=== Score Columns After Cleaning ===")
clean_score_cols = [col + '_clean' for col in score_columns]

for col in clean_score_cols:
    print(f"\n{col}:")
    print(df[col].describe())
    print(f"NaN count: {df[col].isna().sum()}")


original_cols = [
    'cc_contractor_sentiment',
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

df = df.drop(columns=original_cols)

df = df.rename(columns={
    'cc_contractor_sentiment_clean': 'cc_contractor_sentiment',
    'cc_contractor_sentiment_start_score_clean': 'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score_clean': 'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score_clean': 'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score_clean': 'cc_contractor_sentiment_issues_score'
})

print("\nSentiment preprocessing completed successfully!")
print(f"Final dataframe shape: {df.shape}")
print("\nClean Sentiment Distribution:")
print(df['cc_contractor_sentiment'].value_counts())


Original shape: (2123, 34)
Rows where sentiment needs imputation (based on scores): 133

Final Sentiment Distribution (only 3 categories):
cc_contractor_sentiment_clean
Satisfied       1094
Neutral          941
Dissatisfied      88
Name: count, dtype: int64

=== Score Columns After Cleaning ===

cc_contractor_sentiment_start_score_clean:
count    2123.000000
mean       52.538860
std        12.455681
min         0.000000
25%        50.000000
50%        50.000000
75%        50.000000
max       100.000000
Name: cc_contractor_sentiment_start_score_clean, dtype: float64
NaN count: 0

cc_contractor_sentiment_end_score_clean:
count    2123.000000
mean       79.107395
std        16.442335
min         0.000000
25%        80.000000
50%        80.000000
75%        90.000000
max       100.000000
Name: cc_contractor_sentiment_end_score_clean, dtype: float64
NaN count: 0

cc_contractor_sentiment_overall_score_clean:
count    2123.000000
mean       81.793217
std        12.318278
min         0.000000


In [47]:
df.shape

(2123, 34)

In [49]:

print("Before cleaning - Shape:", df.shape)
def clean_yes_no(x):
    if pd.isna(x):
        return "No"

    x_str = str(x).strip().lower()

    if x_str in ['yes', 'y']:
        return "Yes"
        return "No"

# Apply cleaning directly to the 5 columns
df['cc_pricing_mentioned'] = df['cc_pricing_mentioned'].apply(clean_yes_no)
df['cc_pricing_sentiment_impact'] = df['cc_pricing_sentiment_impact'].apply(clean_yes_no)
df['cc_refund_discussed'] = df['cc_refund_discussed'].apply(clean_yes_no)
df['cc_contractor_suggest_leave'] = df['cc_contractor_suggest_leave'].apply(clean_yes_no)
df['cc_contractor_complained'] = df['cc_contractor_complained'].apply(clean_yes_no)


for col in ['cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
            'cc_refund_discussed', 'cc_contractor_suggest_leave',
            'cc_contractor_complained']:
    print(f"{col}:")
    print(df[col].value_counts(dropna=False))
    print("-" * 50)

print(f"\n Cleaning done directly on 'df' dataframe!")
print(f"Current shape: {df.shape}")

Before cleaning - Shape: (2123, 34)

=== AFTER CLEANING THESE 5 COLUMNS ===

cc_pricing_mentioned:
cc_pricing_mentioned
No     1766
Yes     357
Name: count, dtype: int64
--------------------------------------------------
cc_pricing_sentiment_impact:
cc_pricing_sentiment_impact
No     1995
Yes     128
Name: count, dtype: int64
--------------------------------------------------
cc_refund_discussed:
cc_refund_discussed
No     2090
Yes      33
Name: count, dtype: int64
--------------------------------------------------
cc_contractor_suggest_leave:
cc_contractor_suggest_leave
No     2014
Yes     109
Name: count, dtype: int64
--------------------------------------------------
cc_contractor_complained:
cc_contractor_complained
No     1940
Yes     183
Name: count, dtype: int64
--------------------------------------------------

✅ Cleaning done directly on 'df' dataframe!
Current shape: (2123, 34)


The data types of all columns in the `df` DataFrame have already been displayed in the output of the previous cell (`p-Zz1jQ-oqJT`) using `df.info()`:

```
<class 'pandas.core.frame.DataFrame'>
Index: 2123 entries, 3 to 90596
Data columns (total 34 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    2123 non-null   object        
 1   Prospect_Renewal_Date                     2123 non-null   datetime64[ns]
 2   days_to_renewal                           2123 non-null   float64       
 3   Call_Date                                 2123 non-null   datetime64[ns]
 4   Direction                                 2123 non-null   object        
 5   cc_care_package                           2123 non-null   object        
 6   cc_care_package_discussed                 2123 non-null   object        
 7   cc_urgency_getting_on_site                2123 non-null   object        
 8   cc_external_consultant                    2123 non-null   object        
 9   cc_agent_cross_sell_attempt               2123 non-null   object        
 10  cc_customer_issues_concerns               2123 non-null   object        
 11  cc_business_struggles_financial_hardship  2123 non-null   object        
 12  cc_call_initiated_by                      2123 non-null   object        
 13  cc_questionnaire_completion               2123 non-null   object        
 14  cc_chasing_response                       2123 non-null   object        
 15  cc_issues_within_questionnaire            2123 non-null   object        
 16  cc_login_issues                           2123 non-null   object        
 17  cc_platform_issues                        2123 non-null   object        
 18  cc_dissatisfaction_time_to_complete       2123 non-null   object        
 19  cc_process_complexity_concerns            2123 non-null   object        
 20  cc_questions_harder_than_expected         2123 non-null   object        
 21  cc_dissatisfaction_support                2123 non-null   object        
 22  cc_pricing_mentioned                      2123 non-null   object        
 23  cc_pricing_sentiment_impact               2123 non-null   object        
 24  cc_refund_discussed                       2123 non-null   object        
 25  cc_contractor_suggest_leave               2123 non-null   object        
 26  cc_contractor_complained                  2123 non-null   object        
 27  Analysed_Call                             2123 non-null   int64         
 28  Call_Year                                 2123 non-null   int64         
 29  cc_contractor_sentiment                   2123 non-null   object        
 30  cc_contractor_sentiment_start_score       2123 non-null   float64       
 31  cc_contractor_sentiment_end_score         2123 non-null   float64       
 32  cc_contractor_sentiment_overall_score     2123 non-null   float64       
 33  cc_contractor_sentiment_issues_score      2123 non-null   float64       
dtypes: datetime64[ns](2), float64(5), int64(2), object(25)
memory usage: 580.5+ KB
```

This output provides a concise summary, including the data types of each column, which fulfills the requirements of the subtask.

In [51]:
boolean_cols = [
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained'
]

for col in boolean_cols:
    df[col] = df[col].map({'Yes': True, 'No': False}).astype(bool)

print("Boolean columns converted:")
for col in boolean_cols:
    print(f"  {col}: {df[col].dtype}")


Boolean columns converted:
  cc_chasing_response: bool
  cc_issues_within_questionnaire: bool
  cc_pricing_mentioned: bool
  cc_pricing_sentiment_impact: bool
  cc_refund_discussed: bool
  cc_contractor_suggest_leave: bool
  cc_contractor_complained: bool


In [52]:
categorical_cols = [
    'Direction',
    'cc_care_package',
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_call_initiated_by',
    'cc_questionnaire_completion',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_contractor_sentiment'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

print("Categorical columns converted:")
for col in categorical_cols:
    print(f"  {col}: {df[col].dtype}")

Categorical columns converted:
  Direction: category
  cc_care_package: category
  cc_care_package_discussed: category
  cc_urgency_getting_on_site: category
  cc_external_consultant: category
  cc_agent_cross_sell_attempt: category
  cc_customer_issues_concerns: category
  cc_business_struggles_financial_hardship: category
  cc_call_initiated_by: category
  cc_questionnaire_completion: category
  cc_login_issues: category
  cc_platform_issues: category
  cc_dissatisfaction_time_to_complete: category
  cc_process_complexity_concerns: category
  cc_questions_harder_than_expected: category
  cc_dissatisfaction_support: category
  cc_contractor_sentiment: category


In [53]:
print("Final DataFrame dtypes after all conversions:")
df.info()

Final DataFrame dtypes after all conversions:
<class 'pandas.core.frame.DataFrame'>
Index: 2123 entries, 3 to 90596
Data columns (total 34 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    2123 non-null   object        
 1   Prospect_Renewal_Date                     2123 non-null   datetime64[ns]
 2   days_to_renewal                           2123 non-null   float64       
 3   Call_Date                                 2123 non-null   datetime64[ns]
 4   Direction                                 2123 non-null   category      
 5   cc_care_package                           2123 non-null   category      
 6   cc_care_package_discussed                 2123 non-null   category      
 7   cc_urgency_getting_on_site                2123 non-null   category      
 8   cc_external_consultant                    2123 non-null   category

In [54]:
df.to_csv('cc_calls_preprocessed.csv', index=False)

from google.colab import files
files.download('cc_calls_preprocessed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>